<a href="https://colab.research.google.com/github/akshayanandraut/colab/blob/main/comfyui_colab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# ComfyUI Photorealistic Indian Influencer Setup
1. Run the Environment Setup.
2. Download the required models.
3. Enter your ngrok token and launch.
```

In [1]:
#@title 1. Environment Setup
import os

USE_GOOGLE_DRIVE = True #@param {type:"boolean"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
else:
    WORKSPACE = "/content/ComfyUI"

if not os.path.exists(WORKSPACE):
    !git clone https://github.com/comfyanonymous/ComfyUI {WORKSPACE}

%cd {WORKSPACE}
!pip install pyngrok xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ComfyUI
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121


In [2]:
#@title 2. Verify Models & Cleanup
import os

%cd {WORKSPACE}
os.makedirs('./models/checkpoints/', exist_ok=True)
os.makedirs('./models/upscale_models/', exist_ok=True)

ckpt_path = './models/checkpoints/realvisxlV50_v50Bakedvae.safetensors'

# Automatic cleanup of corrupt placeholders
if os.path.exists(ckpt_path):
    size = os.path.getsize(ckpt_path)
    if size < 1000000: # Less than 1MB is definitely a failed download
        print(f"☑️ Removing corrupt file placeholder ({size} bytes)...")
        os.remove(ckpt_path)
        print("✅ Done. Please upload the full 6.5GB model to /ComfyUI/models/checkpoints/ now.")
    else:
        print(f"✅ RealVisXL V5.0 detected ({size / (1024**3):.2f} GB).")
else:
    print("⌛ RealVisXL not found. Waiting for manual upload to: ComfyUI/models/checkpoints/")

# 4x-UltraSharp
print("\nChecking 4x-UltraSharp...")
!wget -c -L "https://huggingface.co/lokCX/4x-UltraSharp/resolve/main/4x-UltraSharp.pth" -P ./models/upscale_models/

/content/drive/MyDrive/ComfyUI
✅ RealVisXL V5.0 detected (6.46 GB).

Checking 4x-UltraSharp...
--2026-07-31 13:39:53--  https://huggingface.co/lokCX/4x-UltraSharp/resolve/main/4x-UltraSharp.pth
Resolving huggingface.co (huggingface.co)... 18.65.14.100, 18.65.14.125, 18.65.14.85, ...
Connecting to huggingface.co (huggingface.co)|18.65.14.100|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /lokCX/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth [following]
--2026-07-31 13:39:53--  https://huggingface.co/lokCX/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/6430096543a53c86b3fcb2a0/080be486975ca8916a1a6fda9e763332b218dbc306ca993f2a029595919e72be?user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%274x-UltraSharp.pth%3B+filename%3D%224x-UltraSharp.pth%22%3B&X-Xet-Cas-Uid=public&Expir

In [4]:
import torch

if torch.cuda.is_available():
    print("✅ CUDA is available! PyTorch detects a GPU.")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    print("❌ CUDA is NOT available. PyTorch does not detect an NVIDIA GPU.")
    print("This is unexpected if your runtime type is set to GPU. Please try 'Runtime -> Factory reset runtime' and then 'Runtime -> Run all'.")

❌ CUDA is NOT available. PyTorch does not detect an NVIDIA GPU.
This is unexpected if your runtime type is set to GPU. Please try 'Runtime -> Factory reset runtime' and then 'Runtime -> Run all'.


In [5]:
import tensorflow as tf

if tf.test.is_gpu_available():
    print("✅ TensorFlow detects a GPU.")
    print("GPU Name:", tf.test.gpu_device_name())
else:
    print("❌ TensorFlow does NOT detect a GPU.")
    print("This confirms the problem is with GPU allocation in the Colab session.")

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


❌ TensorFlow does NOT detect a GPU.
This confirms the problem is with GPU allocation in the Colab session.


In [6]:
# This command typically shows the runtime type selected, but we've seen it can be misleading if PyTorch doesn't pick it up.
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [3]:
#@title 3. Launch ComfyUI via Ngrok (with Real-time Logs)
from pyngrok import ngrok
import threading
import subprocess
import sys
import os

# Paste your token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "3HG2NXigok7WNH2YMXhlymV3M1j_2YP3SwKDSroFHjKj87L3B" #@param {type:"string"}
PORT = 8188

def start_ngrok(port):
    if not NGROK_AUTH_TOKEN:
        print("❌ ERROR: Please enter your ngrok token!")
        return
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    ngrok.kill()
    url = ngrok.connect(port, "http").public_url
    print(f"\n[Ready] URL: {url}\n")

def run_comfy():
    os.chdir(WORKSPACE)
    # Bound to 0.0.0.0 for ngrok compatibility and explicitly set cuda-device 0
    process = subprocess.Popen(
        [sys.executable, "main.py", "--listen", "0.0.0.0", "--port", str(PORT), "--enable-cors-header", "--force-fp16", "--highvram", "--cuda-device", "0"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in process.stdout:
        print(line, end="")

# Start tunnel
if NGROK_AUTH_TOKEN:
    threading.Thread(target=start_ngrok, args=(PORT,), daemon=True).start()

# Start ComfyUI with logging
run_comfy()

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] Set cuda device to: 0

[Ready] URL: https://silk-greeter-craziness.ngrok-free.dev

[WARNING] WARNING: You need pytorch with cu130 or higher to use optimized CUDA operations.
[INFO] Found comfy_kitchen backend triton: {'available': False, 'disabled': True, 'unavailable_reason': 'Neither CUDA nor XPU available on this system', 'capabilities': []}
[INFO] Found comfy_kitchen backend eager: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'convrot_w4a4_linear', 'dequantize_convrot_w4a4_we